In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import json
import time

In [ ]:
def process_action_trace(trace_file_path: Path) -> list:
    """
    Processes the trace.trace file to extract a simplified sequence of user actions.

    Args:
        trace_file_path: Path to the trace.trace file.

    Returns:
        A list of dictionaries, each representing a condensed user action.
    """
    workflow_steps = []
    calls_in_progress = {}

    with open(trace_file_path, "r") as f:
        for line in f:
            try:
                event = json.loads(line)
                call_id = event.get("callId")
                event_type = event.get("type")

                if event_type == "before":
                    calls_in_progress[call_id] = event

                elif event_type == "after" and call_id in calls_in_progress:
                    before_event = calls_in_progress.pop(call_id)

                    # We only care about primary user-facing actions
                    action_class = before_event.get("class")
                    if action_class not in ["Frame", "Page", "Locator"]:
                        continue

                    action_method = before_event.get("method")
                    if action_method not in [
                        "goto",
                        "click",
                        "fill",
                        "press",
                        "selectOption",
                        "check",
                        "uncheck",
                    ]:
                        continue

                    step = {
                        "action": action_method,
                        "target": before_event.get("params", {}).get("url")
                        or before_event.get("params", {}).get("selector"),
                        "status": "Success",
                        "details": {},
                        "timestamp": before_event.get(
                            "time", 0
                        ),  # Add timestamp for ordering
                    }

                    if action_method == "fill":
                        step["value"] = before_event.get("params", {}).get("value")
                    elif action_method == "selectOption":
                        step["value"] = before_event.get("params", {}).get("values")
                    elif action_method == "press":
                        step["key"] = before_event.get("params", {}).get("key")

                    if "error" in event:
                        step["status"] = "Failed"
                        step["details"]["error"] = event["error"].get(
                            "message", "Unknown Error"
                        )

                    workflow_steps.append(step)

            except json.JSONDecodeError:
                continue  # Skip malformed lines

    # Sort by timestamp to ensure proper ordering
    workflow_steps.sort(key=lambda x: x.get("timestamp", 0))
    return workflow_steps


def process_network_trace(network_file_path: Path) -> list:
    """
    Processes the trace.network file to extract key navigation events.

    Args:
        network_file_path: Path to the trace.network file.

    Returns:
        A list of dictionaries, each representing a key network request.
    """
    navigation_events = []
    # Mime types to ignore (static assets, trackers, etc.)
    IGNORE_MIME_TYPES = [
        "image/",
        "text/css",
        "font/",
        "application/javascript",
        "application/x-javascript",
        "text/javascript",
    ]

    # Common tracker/analytics domains to ignore
    IGNORE_DOMAINS = [
        "google-analytics.com",
        "googletagmanager.com",
        "facebook.com",
        "doubleclick.net",
        "googlesyndication.com",
    ]

    with open(network_file_path, "r") as f:
        for line in f:
            try:
                resource = json.loads(line).get("snapshot", {})
                request = resource.get("request", {})
                response = resource.get("response", {})

                url = request.get("url", "")
                mime_type = response.get("content", {}).get("mimeType", "")

                # Skip if it's a type we want to ignore
                if any(ignore in mime_type for ignore in IGNORE_MIME_TYPES):
                    continue

                # Skip known tracker domains
                if any(domain in url for domain in IGNORE_DOMAINS):
                    continue

                # We are primarily interested in document loads, API calls, and form submissions
                if (
                    response.get("status", 0) > 0
                ):  # Exclude failed/aborted requests with status -1
                    event = {
                        "url": url,
                        "method": request.get("method"),
                        "status": response.get("status"),
                        "redirect_url": response.get("redirectURL") or None,
                        "mime_type": mime_type,
                        "size": response.get("content", {}).get("size", 0),
                        "timestamp": resource.get("startedDateTime"),  # Add timing info
                    }

                    # Add request body for POST/PUT requests (useful for form submissions)
                    if request.get("method") in [
                        "POST",
                        "PUT",
                        "PATCH",
                    ] and request.get("postData"):
                        post_data = request["postData"]
                        if (
                            post_data.get("mimeType")
                            == "application/x-www-form-urlencoded"
                        ):
                            # Parse form data
                            event["form_data"] = {
                                param.get("name"): param.get("value")
                                for param in post_data.get("params", [])
                            }
                        elif (
                            post_data.get("text") and len(post_data["text"]) < 1000
                        ):  # Limit size
                            event["request_body"] = post_data["text"]

                    navigation_events.append(event)

            except (json.JSONDecodeError, KeyError):
                continue

    # Sort by timestamp if available
    navigation_events.sort(key=lambda x: x.get("timestamp") or "")
    return navigation_events


def extract_scraping_patterns(workflow_steps: list, navigation_events: list) -> dict:
    """
    Extract common web scraping patterns from the trace data.

    Returns:
        Dictionary containing identified patterns and metadata.
    """
    patterns = {
        "login_detected": False,
        "form_submissions": 0,
        "navigation_count": 0,
        "unique_domains": set(),
        "api_endpoints": [],
        "data_extraction_indicators": [],
    }

    # Analyze workflow steps
    for step in workflow_steps:
        if step["action"] == "goto":
            patterns["navigation_count"] += 1
        elif step["action"] == "fill":
            if any(
                keyword in str(step.get("target", "")).lower()
                for keyword in ["password", "login", "email", "username"]
            ):
                patterns["login_detected"] = True

    # Analyze network events
    for event in navigation_events:
        url = event["url"]
        try:
            domain = urlparse(url).netloc
            patterns["unique_domains"].add(domain)
        except Exception as e:
            pass

        # Detect API endpoints
        if any(
            indicator in url.lower()
            for indicator in ["/api/", "/rest/", ".json", "/graphql"]
        ):
            patterns["api_endpoints"].append(url)

        # Count form submissions
        if event["method"] in ["POST", "PUT", "PATCH"]:
            patterns["form_submissions"] += 1

    # Convert set to list for JSON serialization
    patterns["unique_domains"] = list(patterns["unique_domains"])

    return patterns


def main():
    """
    Process all Playwright trace files in the data/traces directory structure.
    """
    

    base_dir = Path("../data/traces")

    if not base_dir.exists():
        print(f"Error: Directory '{base_dir}' not found")
        return

    # Find all trace directories
    trace_dirs = []
    for trace_dir in base_dir.iterdir():
        if trace_dir.is_dir():
            action_file = trace_dir / "traces" / "trace.trace"
            network_file = trace_dir / "traces" / "trace.network"

            if action_file.exists() and network_file.exists():
                trace_dirs.append(trace_dir)

    if not trace_dirs:
        print(f"No valid trace directories found in '{base_dir}'")
        return

    print(f"Found {len(trace_dirs)} trace directories to process...")

    processed_count = 0
    failed_count = 0

    for trace_dir in trace_dirs:
        try:
            print(f"Processing {trace_dir.name}...")

            action_file = trace_dir / "traces" / "trace.trace"
            network_file = trace_dir / "traces" / "trace.network"
            output_file = trace_dir / "trace.json"

            # Process traces
            workflow = process_action_trace(action_file)
            navigations = process_network_trace(network_file)

            # Extract scraping patterns
            patterns = extract_scraping_patterns(workflow, navigations)

            # Create condensed trace with metadata
            condensed_trace = {
                "metadata": {
                    "trace_name": trace_dir.name,
                    "processed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "workflow_steps_count": len(workflow),
                    "navigation_events_count": len(navigations),
                },
                "scraping_patterns": patterns,
                "workflow_steps": workflow,
                "navigation_events": navigations,
            }

            # Save condensed trace
            with open(output_file, "w") as f:
                json.dump(condensed_trace, f, indent=2, default=str)

            processed_count += 1
            print(f"  ✅ Saved to {output_file}")

        except Exception as e:
            failed_count += 1
            print(f"  ❌ Failed to process {trace_dir.name}: {e}")

    print(f"\n📊 Summary:")
    print(f"  Successfully processed: {processed_count}")
    print(f"  Failed: {failed_count}")
    print(f"  Total: {len(trace_dirs)}")


if __name__ == "__main__":
    main()
